# 极简 Event Tensor Compiler 实现

本文是对于 `Event Tensor: A Unified Abstraction for Compiling Dynamic Megakernel` 论文的一个极简实现，目的是为了理解论文里核心的技术细节。我将沿着 `Event Tensor` 论文里一个`row-sum`的例子进行讲解，主要分三部分：

1. Event Tensor 基本抽象、最小 runtime primitive 设计，以及本文如何表示依赖关系并进行代码生成
2. Static Scheduling 的逻辑，以及如何用多面体编译技术来实现并实现 static task scheduler以及代码生成
3. Dynamic Scheduling 的逻辑，以及如何实现 dynamic task scheduler 和代码生成


本文只追求核心逻辑的实现和演示，在lower、codegen过程中一些零碎的代码都放在 `./etensor` 目录，读者们有兴趣可以自行查看。


In [ ]:
from pathlib import Path
from dataclasses import dataclass
import tempfile

from IPython.display import Markdown, display
import isl
import numpy as np
import torch
import tilelang.language as T
import tvm_ffi

from etensor import codegen as base_codegen
from etensor import tutorial_compiler as etc
from etensor import tutorial_support


def code_block(source: str, lang: str = "cpp", limit: int | None = None) -> None:
    text = source if limit is None else source[:limit]
    display(Markdown(f"```{lang}\n{text}\n```"))


## 1. row-sum 和 Event Tensor

![](tmp/event-tensor-paper/figures_png/fig3_event_sample.png)

论文里的 row-sum 例子讲的是对于一个列数量为动态的Tensor在行上进行两阶段的reduce，传统的方法是需要分别launch两个kernel，而event tensor的方法是把这两个kernel的依赖关系表达出来，然后让编译器去调度和生成代码，实现在一个kernel通过遵守依赖的方式来完成计算。

这里的 row-sum 可以拆成两步, 第一阶段对 `A[n*32, 128]` 在行上按32为tile进行reduce，得到一个 `B[n*32, 4]` 的 Tensor：

$$
B[i, j] = \sum_{k \in [j \cdot 32, j \cdot 32 + 32)} A[i, k]
$$

第二个阶段对 `B[n*32, 4]` 在行上进行reduce，得到一个 `C[n*32]` 的 Tensor：

$$
C[i] = \sum_{j \in [0, 4)} B[i, j]
$$

执行实例 `final_sum(i)` 只依赖 `partial_sum(i, 0) ... partial_sum(i, 3)`，并不依赖别的行，用依赖的方式来表示为：

$$
P[i, j] \rightarrow F[i] \\
\text{P is partial\_sum, F is final\_sum} 
$$


此时把这个依赖关系用一个整型tensor来表示，也就是 Event Tensor：

$$
E[i] = 4 \ \ \\
\text{E is event tensor} 
$$

它的初始值为 4，因为同一个 `F[i]` 要等 4 个partial task 都完成。他的大小会随着 `i` 动态变化。


### 1.1 Event Tensor 实现原理

Event Tensor 的一个元素，本质上就是一个 counter-based synchronization object，通过counter的值来表达同步状态。因此Event Tensor 的 primitive 就是对这个 counter 的操作：

- `notify()`：对 counter 做 atomic decrement。
- `notify_and_ready()`：atomic decrement，并返回这个 event 是否刚好 ready，也就是旧值是否为 1。
- `wait()`：反复读取 counter，直到它变成 0。

我为了保持前端写数组访问的形式，将 API 接口参数设计成 `int* counter` 的形式。所以调用primitive的代码会是这样：
```cpp
etensor::notify(&E0[i]);
etensor::wait(&E0[i]);
```

具体的实现如下:


In [ ]:
code_block(Path("./etensor/include/etensor.cuh").read_text(), "cpp")


### 1.2 实现 device function

在论文的例子中，需要有两个 device function 作为 `partial_sum P` 和 `final_sum F`这两个task，每个 task 都会调用 event tensor 的 primitive 来实现同步。 我这里取巧的使用 TileLang 来实现这两个 Task，因为可以借助 TileLang 生成 CUDA source 的能力，把每个 task 编译成 `__device__` 函数，然后塞进 megakernel。


In [ ]:
M_TILE = 32
K_TILE = 32
K_SPLIT = 4
K = K_TILE * K_SPLIT


def partial_sum(n: int):
    @T.prim_func
    def partial_task(
        A: T.Tensor((n * M_TILE, K), T.float32),
        B: T.Tensor((n * M_TILE, K_SPLIT), T.float32),
        i: T.int32,
        j: T.int32,
    ):
        with T.Kernel(1, threads=128):
            for r in T.Parallel(M_TILE):
                row = i * M_TILE + r
                acc = T.alloc_local((1,), T.float32)
                acc[0] = 0.0
                for k in T.serial(K_TILE):
                    acc[0] += A[row, j * K_TILE + k]
                B[row, j] = acc[0]

    return partial_task


def final_sum(n: int):
    @T.prim_func
    def final_task(
        B: T.Tensor((n * M_TILE, K_SPLIT), T.float32),
        C: T.Tensor((n * M_TILE,), T.float32),
        i: T.int32,
    ):
        with T.Kernel(1, threads=128):
            for r in T.Parallel(M_TILE):
                row = i * M_TILE + r
                acc = T.alloc_local((1,), T.float32)
                acc[0] = 0.0
                for j in T.serial(K_SPLIT):
                    acc[0] += B[row, j]
                C[row] = acc[0]

    return final_task


### 1.3 自定义 TaskGraph IR

论文中编写了一套 `graph function` 的前端来描述 task 和 event 以及它们之间的关系。我们选择最省力的方式，自定义一个很小的 `TaskGraph` IR，只保留教程需要的几个概念：

- `TaskGraph` 包含 buffer、statement、dependence、schedule 和 placement
  - `Tensor` 描述 tensor 的 shape 和 dtype
  - `Statement` 描述一个 task 的执行点和它的计算逻辑
  - `Dependence` 描述 task 之间的依赖关系
  - `Schedule` 描述 task 的执行顺序
  - `Placement` 描述 task 执行的设备
- `Context` 在jit的时提供静态信息，例如这里把 `n` 固定成 8


In [ ]:
@dataclass(frozen=True)
class Tensor:
    domain: isl.set
    dtype: str


@dataclass(frozen=True)
class Statement:
    domain: isl.set
    primfunc: object
    buffers: tuple[str, ...]
    indices: tuple[str, ...]


@dataclass(frozen=True)
class TaskGraph:
    buffers: dict[str, Tensor]
    statements: dict[str, Statement]
    dependence: isl.union_map
    schedule: isl.union_map
    placement: isl.union_map
    context: isl.set | None = None

    def with_context(self, value):
        return value if self.context is None else value.intersect_params(self.context)

    def statement_domain(self, name: str) -> isl.set:
        return self.with_context(self.statements[name].domain)

    def dependence_map(self) -> isl.union_map:
        return self.with_context(self.dependence)

    def schedule_map(self) -> isl.union_map:
        return self.with_context(self.schedule)

    def placement_map(self) -> isl.union_map:
        return self.with_context(self.placement)


接下来我们把论文里 row-sum 例子对应的 TaskGraph IR 构造出来，并且保留它dynamic shape的能力：

In [ ]:
def get_task_graph(n: int) -> TaskGraph:
    buffers = {
        "A": Tensor(
            domain=isl.set(f"[n] -> {{ A[m, k] : 0 <= m < {M_TILE}n and 0 <= k < {K} }}"),
            dtype="float",
        ),
        "B": Tensor(
            domain=isl.set(f"[n] -> {{ B[m, j] : 0 <= m < {M_TILE}n and 0 <= j < 4 }}"),
            dtype="float",
        ),
        "C": Tensor(
            domain=isl.set(f"[n] -> {{ C[m] : 0 <= m < {M_TILE}n }}"),
            dtype="float",
        ),
    }
    statements = {
        "P": Statement(
            domain=isl.set(f"[n] -> {{ P[i, j] : 0 <= i < n and 0 <= j < {K_SPLIT} }}"),
            primfunc=partial_sum(n),
            buffers=("A", "B"),
            indices=("i", "j"),
        ),
        "F": Statement(
            domain=isl.set(f"[n] -> {{ F[i] : 0 <= i < n }}"),
            primfunc=final_sum(n),
            buffers=("B", "C"),
            indices=("i",),
        ),
    }
    dependence = isl.union_map(
        f"[n] -> {{ P[i, j] -> F[i] : 0 <= i < n and 0 <= j < {K_SPLIT} }}"
    )
    schedule = isl.union_map(
        f"[n] -> {{ P[i, j] -> [i, 0, j] : 0 <= i < n and 0 <= j < {K_SPLIT}; F[i] -> [i, 1, 0] : 0 <= i < n }}"
    )
    placement = isl.union_map(
        f"[n] -> {{ [i, t, j] -> BlockIdx[x, y] : x = i and 0 <= i < n and j = y and 0 <= j < {K_SPLIT} }}"
    )
    return TaskGraph(
        buffers=buffers,
        statements=statements,
        dependence=dependence,
        schedule=schedule,
        placement=placement,
        context=isl.set(f"[n] -> {{ : n = {n} }}"),
    )


实例化一个n=8的 TaskGraph：



In [ ]:
graph = get_task_graph(8)
graph

拿到 task graph 后，我们需要把task间的依赖关系转换为 event tensor 的抽象，我在这里引入 `EventInfo` 的数据结构，它是从 TaskGraph 里的 dependence 衍生出来的一个数据结构，专门用来描述 event tensor 相关信息。

这里dependence是他的对应的依赖关系，这里的domain代表了event tensor的shape，init_values这里表示初始值，也就是consumer需要等待多少个producer完成。这里我特意做成了一个ndarray，是考虑到复杂的例子下，不同的consumer可能需要等待不同数量的producer完成。 另外这里producer和consumer只是单纯的匹配task name，最后的notify_access和wait_access实际是为了描述Producer和Consumer应该访问event tensor的哪个元素，后续codegen会用到。

In [ ]:
@dataclass(frozen=True)
class EventInfo:
    name: str
    dependence: isl.map
    domain: isl.set
    init_values: np.ndarray
    producer: str
    consumer: str
    notify_access: isl.map
    wait_access: isl.map


下面需要从TaskGraph IR 里把这些信息提取出来，构造出 `EventInfo`，其实很简单，每个dependence的 range就是 event tensor的shape，然后在每个dependence上统计一下每个consumer需要等待多少个producer，就得到了event tensor的初始值。最后把producer和consumer的task name记录一下，构建出 `EventInfo`。

In [ ]:
def derive_event_infos(graph: TaskGraph) -> tuple[EventInfo, ...]:
    events = []
    maps = graph.dependence_map().get_map_list()
    for i in range(maps.n_map()):
        dep = maps.get_at(i)
        name = f"E{i}"
        domain = dep.range()
        lows = [domain.dim_min_val(dim).get_num_si() for dim in range(domain.dim(isl.dim_type.SET))]
        shape = tuple(
            domain.dim_max_val(dim).get_num_si() - lows[dim] + 1
            for dim in range(domain.dim(isl.dim_type.SET))
        )
        init_values = np.zeros(shape, dtype=np.int32)

        wait_access = domain.identity().set_tuple_name(isl.dim_type.OUT, name)
        pma = wait_access.as_pw_multi_aff()
        for dim, low in enumerate(lows):
            if low != 0:
                pma = pma.set_at(dim, pma.at(dim).add_constant(-low))
        wait_access = pma.as_map()

        def fill(point: isl.point) -> None:
            index = tuple(
                point.get_coordinate_val(isl.dim_type.SET, dim).get_num_si() - lows[dim]
                for dim in range(point.dim(isl.dim_type.SET))
            )
            init_values[index] = dep.intersect_range(point.to_set()).domain().count_val().get_num_si()

        domain.foreach_point(fill)
        events.append(
            EventInfo(
                name=name,
                dependence=dep,
                domain=domain,
                init_values=init_values,
                producer=dep.get_tuple_name(isl.dim_type.IN),
                consumer=dep.get_tuple_name(isl.dim_type.OUT),
                notify_access=dep.apply_range(wait_access),
                wait_access=wait_access,
            )
        )
    return tuple(events)


events = derive_event_infos(graph)
events


### 1.4 从TaskGraph到CodeGen

得到TaskGraph后，我打算使用最naive的方式将它所表示的逻辑进行代码生成，那么就是复用isl的 AST builder，通过marker的方式标记每个 statement instance， 当 statement 打印时，同时打印对应的 `notify` 或 `wait` 。

这里最重要的是 isl AST builder 的两个 hook：

- `at_each_domain`：isl 每生成一个 statement instance，就调用它。我们在这里知道当前打印的是 `P` 还是 `F`。
- `print_user`：真正把这个 statement 打印成 CUDA 代码。我们在这里把 `wait/task/notify` 按顺序输出。

其中 marker 本身仍然只表示“这里要插同步”。在之前的 `EventInfo` 的计算过程中，我们已经把 `P[i, j] -> F[i]` 这个依赖关系转化成了两条 access map，因此具体访问哪个 `E[...]`，由当前 statement instance apply 对应 access map 得到。

```text
notify_access: P[i, j] -> E0[i]
wait_access:   F[i]    -> E0[i]
```

下面的 `render_block_schedule` 就是这个过程的核心：先用 isl 生成 block 内的 statement AST，再在 `at_each_domain` 里为当前 statement 算出 wait/notify 的 event access，最后通过 `print_user` 把普通 task call 替换成 `wait -> task -> notify` 的 CUDA 代码。


In [ ]:
def render_block_schedule(graph: TaskGraph, events: tuple[EventInfo, ...]) -> str:
    schedule = base_codegen.build_block_schedule_tree(graph, events)
    user_infos: dict[int, base_codegen.UserPrintInfo] = {}

    def after_mark_callback(node: isl.ast_node_mark, build: isl.ast_build) -> isl.ast_node:
        child = node.node()
        return isl.ast_node_block(isl.ast_node_list(isl.ast_node(child)))

    def at_each_domain(node: isl.ast_node_user, build: isl.ast_build) -> isl.ast_node:
        expr = node.expr()
        statement = expr.get_arg(0).get_id().get_name()
        waits = tuple(
            base_codegen.EventCall(
                event.name,
                base_codegen._event_args_for_current_instance(event.wait_access, build),
            )
            for event in events
            if statement == event.consumer
        )
        notifies = tuple(
            base_codegen.EventCall(
                event.name,
                base_codegen._event_args_for_current_instance(event.notify_access, build),
            )
            for event in events
            if statement == event.producer
        )

        annotation = isl.id(f"{statement}_{len(user_infos)}")
        user_infos[annotation.ptr] = base_codegen.UserPrintInfo(statement, waits, notifies)
        return node.set_annotation(annotation)

    def print_user(printer, options, node):
        expr = node.expr()
        info = user_infos[node.annotation().ptr]

        for event_call in info.waits:
            printer = base_codegen._print_event_call(printer, event_call, "wait")
        if info.waits:
            printer.start_line(); printer.print_str("__syncthreads();"); printer.end_line()

        printer.start_line()
        printer.print_str(f"{info.statement}(")
        for i in range(1, expr.get_n_arg()):
            if i != 1:
                printer.print_str(", ")
            printer.print_ast_expr(expr.get_arg(i))
        printer.print_str(");")
        printer.end_line()

        if info.notifies:
            printer.start_line(); printer.print_str("__syncthreads();"); printer.end_line()
            for event_call in info.notifies:
                printer = base_codegen._print_event_call(printer, event_call, "notify")
        return printer

    fd, raw_path = tempfile.mkstemp(suffix=".c")
    path = Path(raw_path)
    try:
        builder = isl.ast_build.from_context(graph.context) if graph.context is not None else isl.ast_build()
        builder = builder.set_after_each_mark(after_mark_callback)
        builder = builder.set_at_each_domain(at_each_domain)
        ast = builder.node_from(schedule)
        printer = isl.printer.to_file_path(str(path)).set_output_format(isl.format.C)
        options = isl.ast_print_options.alloc().set_print_user(print_user)
        ast.print(printer, options).flush()
        return path.read_text().strip()
    finally:
        path.unlink(missing_ok=True)


schedule_code = render_block_schedule(graph, events)


In [ ]:
code_block(schedule_code, "cpp")

当核心代码已经构建，剩下就是添加 host 代码、插入头文件、渲染模板以及编译动态库等细节。这些都放在 `./etensor` 目录里了，这里直接调用 `render_base_cuda_source` 来生成最终的代码：

In [ ]:
base_cuda = etc.render_base_cuda_source(graph, events, schedule_code=schedule_code)
code_block(base_cuda, "cpp", limit=4000)

## 2. Static scheduling：从逻辑 schedule 到 processor-time schedule

接下来进入论文里的 static scheduling。

`summary.md` 对这一节的概括是：静态调度会在 launch 前显式把 task 分配到不同 SM 上，将多个 device function 融合在一起；每个 task 会预先分配到某个 SM 的 task queue 中。

更学术一点说，这里做的不是普通 loop transformation，而是一次 **resource-constrained schedule lowering**：原始 graph 里的 schedule 只规定 logical task 之间的相对执行顺序，而 static scheduling 进一步把这些 logical task 映射到一个有限的处理器集合上，也就是 persistent CTA/SM worker。

优化前，schedule 的 target 还是逻辑执行空间：

```text
task instance -> logical time / tile coordinate
blockIdx      -> logical tile coordinate
```

优化后，schedule 被 lowering 到 processor-time space：

```text
task instance -> (worker, local_step)
blockIdx.x    -> worker id
local_step    -> worker 本地 queue 中的顺序
```

因此，`blockIdx` 不再表示原始 computation domain 里的 tile 坐标，而是表示一个物理执行资源。原来的逻辑 tile 坐标被 materialize 到 task descriptor 中，由 scheduler 在循环里取出。

这一节我先用 isl 把这个变换写成几个关系：

```text
task instance -> logical time -> linear time -> resource(worker, step)
```

其中 `worker` 是 processor dimension，`step` 是该 processor 上的 local time dimension。这和 `etensor/demo/resource_schedule_poc.py` 里的 POC 是同一个思路。最后为了生成 CUDA constant array，我们再枚举这些点，把每个 worker 的 queue materialize 出来。


In [ ]:
N = 8
WORKERS = 4
TOTAL_TASKS = N * (K_SPLIT + 1)
STEPS = (TOTAL_TASKS + WORKERS - 1) // WORKERS

# 1. 先把 task instance 映射到逻辑时间坐标。
# P[i, j] 在 phase 0，F[i] 在 phase 1。
task_to_time = isl.union_map(
    f"{{ "
    f"P[i, j] -> Time[i, 0, j] : 0 <= i < {N} and 0 <= j < {K_SPLIT}; "
    f"F[i] -> Time[i, 1, 0] : 0 <= i < {N} "
    f"}}"
)

# 2. 把多维逻辑时间线性化。
# 对每个 i，有 K_SPLIT 个 P，再接一个 F，所以一行有 K_SPLIT + 1 个 task。
time_to_linear = isl.map(
    f"{{ Time[i, phase, j] -> T[t] : "
    f"t = i * {K_SPLIT + 1} + phase * {K_SPLIT} + j "
    f"}}"
)

# 3. 把线性时间映射到固定数量的 worker 上。
# 这里就是最简单的 modulo schedule: worker = t % WORKERS。
linear_to_resource = isl.map(
    f"{{ T[t] -> R[worker, step] : "
    f"t = step * {WORKERS} + worker and 0 <= worker < {WORKERS} "
    f"}}"
)

task_to_resource = task_to_time.apply_range(time_to_linear).apply_range(linear_to_resource)
resource_to_task = task_to_resource.reverse()

print("task_to_resource:")
print(task_to_resource)
print()
print("resource_to_task:")
print(resource_to_task)


In [ ]:
@dataclass(frozen=True)
class TaskInstance:
    statement: str
    indices: tuple[int, ...]
    schedule_key: tuple[int, ...]


def point_tuple(point: isl.point) -> tuple[int, ...]:
    return tuple(
        int(str(point.get_coordinate_val(isl.dim_type.SET, dim)))
        for dim in range(point.dim(isl.dim_type.SET))
    )


def enumerate_task_instances(graph: TaskGraph) -> tuple[TaskInstance, ...]:
    schedule = graph.schedule_map()
    instances = []

    for statement in graph.statements:
        domain = graph.statement_domain(statement)

        def visit(point: isl.point) -> None:
            schedule_points = []
            point.to_set().apply(schedule).foreach_point(lambda p: schedule_points.append(point_tuple(p)))
            instances.append(TaskInstance(statement, point_tuple(point), schedule_points[0]))

        domain.foreach_point(visit)

    return tuple(sorted(instances, key=lambda x: x.schedule_key))


def modulo_static_queues(instances: tuple[TaskInstance, ...], worker_count: int):
    queues = [[] for _ in range(worker_count)]
    for logical_time, instance in enumerate(instances):
        worker = logical_time % worker_count
        queues[worker].append(instance)
    return queues


logical_trace = enumerate_task_instances(graph)
static_queues = modulo_static_queues(logical_trace, worker_count=WORKERS)

print("logical trace:")
print([(x.statement, x.indices, x.schedule_key) for x in logical_trace[:12]])
print()
for worker, queue in enumerate(static_queues):
    print("worker", worker, [(x.statement, x.indices) for x in queue[:8]])


上面是多面体关系和 Python 枚举得到的 queue。接下来要把它 lowering 成 CUDA 里能直接消费的数据结构。

static scheduler 需要三个 constant array：

```cpp
static_task_indices[]  // 所有 task 坐标展平后的数组，例如 P 需要两个 index，F 需要一个 index
static_tasks[]         // 每个 task 的 task_type 和 index_begin
static_queues[]        // 每个 worker 的 task_begin/task_end
```

可以把它理解成一个扁平的 CSR-like 表示：

```text
static_queues[worker] -> task range
static_tasks[task]    -> task type + index offset
static_task_indices   -> flattened logical coordinates
```

具体执行时：

- `static_queues[blockIdx.x]` 告诉当前 worker 应该消费哪一段 task。
- `static_tasks[task_pos]` 告诉 scheduler 当前 task 是 `P` 还是 `F`，以及它的 index 从哪里开始。
- `static_task_indices + index_begin` 就是 task 的逻辑坐标指针。对 `P` 来说可以读 `task_idx[0], task_idx[1]`；对 `F` 来说只读 `task_idx[0]`。

persistent kernel 里面每个 worker 做的事情很简单：

```cpp
StaticTaskScheduler scheduler;
scheduler.init();
while (scheduler.valid()) {
  const int* task_idx = scheduler.indices();
  switch (scheduler.type()) { ... }
}
```

注意，static queue 里可以提前放 `F[i]`。如果它被执行得太早，`wait(E[i])` 会挡住它。这个语义和论文里的静态调度是一致的：调度顺序提前决定，依赖正确性仍然由 Event Tensor 保证。

下面直接看 runtime scheduler 的实现。它没有全局抢占逻辑，也没有 ready queue，只是每个 worker 顺序扫描自己那段 static queue。


In [ ]:
code_block(Path("./etensor/include/static_tile_scheduler.cuh").read_text(), "cpp")


In [ ]:
@dataclass(frozen=True)
class StaticQueue:
    task_kinds: tuple[int, ...]
    task_indices: tuple[int, ...]


@dataclass(frozen=True)
class StaticScheduledGraph:
    source: TaskGraph
    events: tuple[EventInfo, ...]
    worker_count: int
    task_kinds: dict[str, int]
    task_index_ranks: tuple[int, ...]
    queues: tuple[StaticQueue, ...]


def static_schedule(graph: TaskGraph, worker_count: int) -> StaticScheduledGraph:
    task_kinds = {name: i for i, name in enumerate(graph.statements)}
    task_index_ranks = tuple(len(statement.indices) for statement in graph.statements.values())
    queues = []
    for queue in modulo_static_queues(enumerate_task_instances(graph), worker_count):
        kinds = tuple(task_kinds[x.statement] for x in queue)
        indices = tuple(v for x in queue for v in x.indices)
        queues.append(StaticQueue(kinds, indices))
    return StaticScheduledGraph(graph, events, worker_count, task_kinds, task_index_ranks, tuple(queues))


static_plan = static_schedule(graph, worker_count=WORKERS)
for array in etc.build_static_queue_arrays(static_plan):
    print(array.name, "=", array.values[:160], "...")

static_cuda = etc.render_static_cuda_source(static_plan)
code_block(static_cuda, "cpp", limit=4000)


## 3. Dynamic scheduling：online processor-time schedule

Dynamic scheduling 这一节按和 static scheduling 类似的方式来拆：先看变换生成什么调度数据，再看 runtime scheduler 怎么消费这些数据。

如果说 static scheduling 是在编译期构造一个完整的 processor-time schedule：

```text
task instance -> (worker, local_step)
```

那么 dynamic scheduling 可以理解成一个 **online schedule construction**：编译器不再提前决定每个 logical task 由哪个 worker 在第几个 local step 执行，而是只生成 task set、依赖触发规则和 scheduler 数据结构；实际的 `(worker, local_step)` 映射在运行时由 pop 操作决定。

换句话说，两者的区别不是有没有 Event Tensor 依赖，而是 processor-time binding 发生在什么时候：

```text
static scheduling  : compile-time binding
dynamic scheduling : runtime binding
```

这个 notebook 里的 row-sum v1 先做一个 stage-level 简化版：

1. 初始化时只 push `P` task set。
2. `P` set 被领取完后 push `F` task set。
3. `F[i]` 内部仍然 wait `E[i]`，保证每一行的依赖正确。

论文里的完整 dynamic scheduling 是 event-triggered：当某个 event counter 变成 0，就把对应 consumer task push 进 scheduler。row-sum v1 还不是 MoE 那种 data-dependent push，但已经展示了关键抽象：`blockIdx.x` 只表示 persistent worker，logical tile index 由 scheduler 在运行时返回。


In [ ]:
@dataclass(frozen=True)
class DynamicTaskSet:
    task_kind: int
    instances: tuple[TaskInstance, ...]
    next_set_id: int = -1


@dataclass(frozen=True)
class DynamicScheduledGraph:
    source: TaskGraph
    events: tuple[EventInfo, ...]
    worker_count: int
    task_kinds: dict[str, int]
    task_index_ranks: tuple[int, ...]
    task_sets: tuple[DynamicTaskSet, ...]
    initial_ready_sets: tuple[int, ...]


def dynamic_schedule(graph: TaskGraph, worker_count: int) -> DynamicScheduledGraph:
    task_kinds = {name: i for i, name in enumerate(graph.statements)}
    task_index_ranks = tuple(len(statement.indices) for statement in graph.statements.values())
    by_statement = {name: [] for name in graph.statements}
    for instance in enumerate_task_instances(graph):
        by_statement[instance.statement].append(instance)

    event = events[0]
    task_sets = (
        DynamicTaskSet(task_kinds[event.producer], tuple(by_statement[event.producer]), next_set_id=1),
        DynamicTaskSet(task_kinds[event.consumer], tuple(by_statement[event.consumer])),
    )
    return DynamicScheduledGraph(graph, events, worker_count, task_kinds, task_index_ranks, task_sets, initial_ready_sets=(0,))


dynamic_plan = dynamic_schedule(graph, worker_count=WORKERS)
for set_id, task_set in enumerate(dynamic_plan.task_sets):
    print("set", set_id, "kind", task_set.task_kind, "tiles", [x.indices for x in task_set.instances[:8]], "next", task_set.next_set_id)


Dynamic scheduler 生成的数据结构也可以拆成两层：

```cpp
dynamic_task_indices[]  // 所有 task set 的 logical coordinates，提前展开好
task_sets[]             // 每个 set 的 task_type/index_rank/index_begin/index_end/next_set_id
```

和 static queue 的区别在于：

- static 是 `worker -> task range`，worker 只消费自己的 queue。
- dynamic 是 `ready queue -> task set -> tile cursor`，多个 worker 可以竞争同一个 ready set。

runtime 里还有几组全局状态：

```cpp
dynamic_set_index_pos[set]  // 每个 task set 当前消费到哪个 index
dynamic_ready_queue[]       // 当前 ready 的 task set id 队列
dynamic_queue_head/tail     // ready queue 的 head/tail
dynamic_queue_lock          // 简单 centralized queue lock
dynamic_tiles_done          // 已经 dispatch 出去的 tile 数
```

这个版本生成的 persistent kernel 主循环是：

```cpp
__shared__ TaskScheduler scheduler;
scheduler.init();
while (scheduler.valid()) {
  const int* task_idx = scheduler.indices();
  switch (scheduler.type()) { ... }
}
```

这里 `scheduler.valid()` 不只是判断条件，它也负责从全局 ready queue 里领取下一个 tile。

下面先看生成的数据结构，再看 dynamic runtime scheduler 的实现。


In [ ]:
for array in etc.build_dynamic_scheduler_arrays(dynamic_plan):
    print(array.name, "=", array.values[:200], "...")

code_block(Path("./etensor/include/dynamic_tile_scheduler.cuh").read_text(), "cpp")


In [ ]:
dynamic_cuda = etc.render_dynamic_cuda_source(dynamic_plan)
code_block(dynamic_cuda, "cpp", limit=4000)


## 小结

到这里，这个教程就形成了三块：

1. **第一个例子**：row-sum、Event Tensor counter runtime、基础 isl AST codegen。
2. **Static scheduling**：把 logical task schedule lowering 成 compile-time processor-time schedule，并 materialize 成 per-worker queue。
3. **Dynamic scheduling**：把 processor-time binding 推迟到运行时，由 ready queue 和 scheduler pop 动态决定 task 由哪个 worker 执行。

这基本对应论文里的主线：Event Tensor 负责紧凑表达细粒度依赖，scheduling transformation 负责把同一个依赖图 lowering 到不同的执行资源模型上。

更形式化地看，Event Tensor graph 给出的是一个带依赖约束的 logical task graph；static scheduling 构造一个离线的资源绑定函数：

```text
logical task -> (worker, local_step)
```

而 dynamic scheduling 则保留 task set 和 dependency trigger，把这个绑定函数的一部分交给 runtime scheduler 在线构造。

所以两者不是谁替代谁，而是同一个 Event Tensor IR 在不同 workload 假设下的两种 lowering：静态调度最小化 runtime overhead，动态调度换取更强的负载均衡和对数据依赖动态性的适应能力。
